# Star Formation

This notebook shows how to render **grid-based snapshots** from Athena++:

QuickStart:

```text
AMR snapshot -> export_volume_vdb_sequence() -> .vdb file -> setup_animation() in Blender
```

Use this when the source data is already volume data, for example Athena++ or another AMR code.


## 1. Export VDB frames

Run this section in a Python 3.11 environment with `yt` and `openvdb` available.


In [ ]:
import sys
sys.path.append("../../..") ## Set it to the path of the AstroVis package if you are running this script outside of the package directory. 

from AstroVis.backend import export_volume_vdb_sequence

input_dir = "Data"
output_dir = "Output"
plot_dir = "Plots"

field_name = "density"
gas_type = "gas"
num_snapshot = 5  #If you want to process all available snapshots, omit this keyword argument.

export_volume_vdb_sequence(input_dir="Data", output_dir="Output", num_snapshot=10,
                            vtype="gas", field="density", log=True,
                            preview_dir="Plots",preview_every=1,
                            zip_output=True, delete_unzipped=True,)


In [ ]:
# Check ds.derived_field_list first when you want to confirm the yt particle type and field name.

import os
import yt

snapshots = sorted([f for f in os.listdir(input_dir) if f.endswith(".athdf")])
ds = yt.load(os.path.join(input_dir, snapshots[0]))
print(ds.derived_field_list)

## 2. Import and render in Blender

Run this section **inside Blender's Python environment**.  
Opening 'Scripting' in the Blender, add a new text block and paste the following script in the text block.  
Press 'Run' button.

In [ ]:
import bpy

import sys
sys.path.append("../..") #Set the path to the AstroVis package 

from AstroVis.api import *

vdb_folder = r"Path/to/your/vdb/folder"  # Replace with the actual path to your VDB folder

scene = SceneManager()
scene.clear_scene(use_empty=False)

species_name = "Star_Formation"  

material = create_field_volume_material(
    species_name,
    field_min=-2, ## Take the value report in the preview output
    field_max=5,
    apply=False,
)[species_name]

setup_animation(data_path=vdb_folder, object=species_name, material=material)
scene.set_camera(location=(10, 10, 10), look_at=(0, 0, 0))

You may run the bash script Rendering.sh directly for rendering figures in headless mode. 
Just change the path of blender to where you install:

```GALLIUM_DRIVER=llvmpipe path/to/your/blender -b Star_Formation.blend -E CYCLES --python blender.py -o "//Blender_Plot/frame_" -s 0 -e 100 -a```

And check the figuresin Blender_Plot.

## Notes

- `ds.field_list` is the first thing to check when you are unsure about the yt field type or the field name.
- `export_volume_sequence()` is the simple one-pass helper: it exports every frame, saves preview images every 10th frame by default, and returns the final global range for Blender.
- Keep the same `field_min` and `field_max` across all frames so the Blender material uses one consistent scale.
- The examples here intentionally use only the minimum useful arguments. For optional controls, see `Documentation/Backend/Volume_Data.md`, `Documentation/Blender/Animation_setup.md`, and `Documentation/Blender/Node_and_shading.md`.
- `setup_volume_animation()` is the lower-level alternative when you want to import the VDB sequence directly instead of using `setup_animation()`.
- The volume material already emits light from the data values, so this example keeps the scene simple and does **not** add extra lights.


## Detail API

1. loads each snapshot with `yt.load()`
2. calls `load_volume()` to build the AMR hierarchy
3. computes the field range and preview slice via `analyze_field_data()`
4. calls `hierarchy_to_multiple_vdbs()` to export each frame to VDB files (`frame_0000/`, `frame_0001/`, ...)
5. saves previews every `preview_every` frames using the final global range
6. prints the final global min/max so the Blender material can reuse the same scale